# Cwiczenie 3 - Pandas (seria 3)

Proszę mieć na uwadze, że zestaw jest celowo ustawiony tak, aby stopniowo budowac kompetencje potrzebne do projektu podsumowujacego 1-4: najpierw czyszczenie danych i selekcja, potem merge/groupby/pivot,
a na koncu proste elementy optymalizacji w Pandas.

Nowe zadania dopinaja luki z calego wykladu 3:
- alignment po indeksie,
- concat z `keys`,
- audyt merge przez `validate` i `indicator`,
- `transform` oraz `filter`,
- `pivot_table` z `fill_value` i `margins`,
- regexy w `.str`,
- lag/shift w szeregach czasowych,
- mini-benchmarki.

W zadaniach z sekcja **Wymagane metody** postaraj sie nie omijac wskazanych narzedzi.


## Proponowany tryb pracy

**CORE (obowiązkowe minimum):**
- 03.01–03.09
- 03.11.1
- 03.07.2 (merge audit) warto zrobić wspólnie z prowadzącym

**ROZSZERZENIE / dla chętnych:**
- 03.10.2 (regex)
- 03.11.2 (lag / shift)
- 03.12.2–03.12.3 (mini-benchmarki)
- sekcja o `category`

Można to potraktować też jak dwie ścieżki.

Zadania z gwiązdką rozszerzające/dla chętnych.

In [3]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def _ensure_core_import() -> Path:
    candidates = [
        Path.cwd(),
    ]
    for candidate in candidates:
        if (candidate / "projekt_1_4_bikeshare_core.py").exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    raise FileNotFoundError("Nie znaleziono pliku projekt_1_4_bikeshare_core.py")


CORE_PATH = _ensure_core_import()

from projekt_1_4_bikeshare_core import make_bikeshare_data, revenue_loop, time_one

pd.set_option("display.max_columns", 30)
np.random.seed(0)

trips, stations, weather = make_bikeshare_data(n_trips=5000, n_stations=18, n_days=14, seed=0)

# pomocnicze kopie robocze
trips_small = trips.copy()
stations_small = stations.copy()
weather_small = weather.copy()

print(f"Core imported from: {CORE_PATH}")
print(trips_small.head(2))
print(stations_small.head(2))
print(weather_small.head(2))

Core imported from: /home/piotr/Documents/zajecia/ML/ML_kurs/3/cwiczenie_3_numpy_full_pack
   trip_id  bike_id            start_ts                        end_ts  \
0        0       70 2024-05-05 09:51:40 2024-05-05 10:25:12.007131760   
1        1       90 2024-05-04 05:34:31 2024-05-04 06:23:50.876575000   

   raw_start_lat  raw_start_lon  raw_end_lat  raw_end_lon user_type  \
0      52.179564      21.011720    52.210428    21.082534    casual   
1      52.119302      20.944685    52.187949    21.019742    casual   

  coupon_code  duration_min_true  distance_km_true  true_start_station_id  \
0   SUMMER10           33.533452          5.918933                      7   
1       bike5          49.331276          9.284501                      3   

   true_end_station_id  
0                   14  
1                    8  
   station_id      station_name district        lat        lon  capacity
0           0   st_00 | praga      Praga  52.155710  20.949305        21
1           1   ST-01 

## Mapa tego notebooka

Ten zestaw najlepiej działa w trzech krokach:

1. **Oczyść i opisz dane**  
   (`select`, `assign`, missing values, stringi)

2. **Połącz dane i policz agregaty**  
   (`merge`, `groupby`, `pivot_table`, `transform`, `validate`)

3. **Zamień tabelę w prostą analizę**  
   (`rolling`, `lag`, krótkie wnioski analityczne)

Na zajęciach warto pilnować jednej zasady:
**najpierw liczba wierszy i klucze, potem dopiero ładne wyniki.**


## 03.01 - Introducing Pandas Objects


### Cwiczenie 3.01.1 - Seria KPI z DataFrame

Napisz funkcje, ktora zbuduje obiekt `pd.Series` z podstawowymi KPI:
- `n_trips`
- `n_unique_bikes`
- `mean_duration_min`
- `share_member`


In [ ]:
Jasne. Tutaj masz wersję TODO z dużo lepszym naprowadzeniem, ale dalej bez gotowego rozwiązania:

def trip_kpi_series(trips: pd.DataFrame) -> pd.Series:
    # TODO:
    # Funkcja ma zwrócić pd.Series z dokładnie 4 KPI w tej kolejności:
    # 1) "n_trips"            -> liczba wszystkich wierszy w tabeli trips
    # 2) "n_unique_bikes"     -> liczba unikalnych rowerów (kolumna "bike_id")
    # 3) "mean_duration_min"  -> średni czas przejazdu w minutach
    # 4) "share_member"       -> udział przejazdów użytkowników typu "member"
    #
    # Wskazówki:
    # - czas przejazdu policz z różnicy:
    #       end_ts - start_ts
    # - obie kolumny czasu warto najpierw zamienić przez pd.to_datetime(...)
    # - różnica dat daje timedelta, więc aby dostać minuty użyj:
    #       .dt.total_seconds() / 60.0
    # - liczba przejazdów:
    #       len(trips)
    # - liczba unikalnych rowerów:
    #       trips["bike_id"].nunique()
    # - średni czas:
    #       duration_min.mean()
    # - udział member:
    #       trips["user_type"].eq("member").mean()
    #   bo True/False w Pandas liczą się jak 1/0
    #
    # Na końcu zbuduj i zwróć pd.Series ze słownika:
    # {
    #     "n_trips": ...,
    #     "n_unique_bikes": ...,
    #     "mean_duration_min": ...,
    #     "share_member": ...,
    # }
    #
    # Uwaga:
    # - zachowaj dokładnie nazwy indeksów i ich kolejność,
    #   bo sprawdza to assert w testach.
    raise NotImplementedError

kpi = trip_kpi_series(trips_small)
assert isinstance(kpi, pd.Series)
assert list(kpi.index) == ["n_trips", "n_unique_bikes", "mean_duration_min", "share_member"]
assert int(kpi["n_trips"]) == len(trips_small)
assert 0.0 <= float(kpi["share_member"]) <= 1.0
kpi


## 03.02 - Data Indexing and Selection


### Cwiczenie 3.02.1 - Selekcja przejazdow w godzinach szczytu

Zwracaj tylko przejazdy rozpoczete:
- rano: od 7:00 do 9:59
- wieczorem: od 16:00 do 18:59

Dodaj kolumne `start_hour`.


In [ ]:
def select_rush_hours(trips: pd.DataFrame) -> pd.DataFrame:
    # TODO pseudokod:
    # 1) skopiuj wejściowy DataFrame
    # 2) wyciągnij godzinę startu z kolumny "start_ts" do "start_hour"
    # 3) zaznacz wiersze, gdzie start_hour jest w [7, 9] lub w [16, 18]  mask = out["start_hour"].between(7, 9) | out["start_hour"].between(16, 18)
    # 4) zwróć tylko te wiersze jako nowy DataFrame #.copy()
    raise NotImplementedError

rush = select_rush_hours(trips_small)
assert isinstance(rush, pd.DataFrame)
assert "start_hour" in rush.columns
assert rush["start_hour"].between(7, 9).all() or rush["start_hour"].between(16, 18).any()
assert set(rush["start_hour"].unique()).issubset(set([7, 8, 9, 16, 17, 18]))
rush.head()


## 03.03 - Operations in Pandas


### Cwiczenie 3.03.1 - Dodaj cechy obliczone wektorowo

Utworz kopie `trips` i dodaj kolumny:
- `duration_min` = roznica `end_ts - start_ts` w minutach
- `is_member` = czy `user_type == "member"`
- `revenue_est` wedlug prostego cennika:
  - member: 1.0 + 0.12 * max(duration_min - 5, 0)
  - casual: 2.5 + 0.23 * max(duration_min - 5, 0)

Bez petli po wierszach.


In [ ]:
def add_trip_features(trips: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Zwróć kopię DataFrame wzbogaconą o nowe cechy przejazdu.
    #
    # Masz dodać co najmniej te kolumny:
    # 1) "duration_min"  -> czas przejazdu w minutach
    # 2) "is_member"     -> czy użytkownik ma typ "member" (True/False)
    # 3) "revenue_est"   -> prosty szacowany przychód z przejazdu
    # 4) "date"          -> data przejazdu obcięta do początku dnia
    #
    # Wymagania:
    # - pracuj wektorowo na całych kolumnach
    # - nie używaj pętli po wierszach
    # - zacznij od kopii danych:
    #       out = trips.copy()
    #
    # Krok 1. Czas przejazdu:
    # - policz różnicę między "end_ts" i "start_ts"
    # - wcześniej zamień obie kolumny przez pd.to_datetime(...)
    # - wynik zamień na minuty przez:
    #       .dt.total_seconds() / 60.0
    # - zapisz do kolumny "duration_min"
    #
    # Krok 2. Flaga użytkownika:
    # - kolumna "is_member" ma być typu bool
    # - sprawdź, czy "user_type" jest równe "member"
    # - przyda się:
    #       out["user_type"].eq("member")
    #
    # Krok 3. Szacowany przychód:
    # - załóż inny koszt bazowy i inną stawkę za minutę
    #   dla member i non-member
    # - wygodnie zrobić to przez np.where(...)
    #
    # Schemat:
    # - base    -> koszt początkowy zależny od "is_member"
    # - per_min -> stawka za minutę zależna od "is_member"
    # - pierwsze 5 minut są "w cenie"
    # - dopiero nadwyżka ponad 5 minut zwiększa przychód
    #
    # Wskazówka:
    # - nadwyżkę minut możesz policzyć przez coś w rodzaju:
    #       maksimum z (duration_min - 5) oraz 0
    # - do tego przyda się np.maximum(...)
    #
    # Ostatecznie:
    #       revenue_est = base + per_min * extra_minutes
    #
    # Krok 4. Data bez godziny:
    # - z "start_ts" utwórz kolumnę "date"
    # - ma zawierać początek dnia dla każdego przejazdu
    # - przyda się:
    #       pd.to_datetime(...).dt.floor("D")
    #
    # Na końcu zwróć out.
    #
    # Uwaga:
    # - testy sprawdzają obecność kolumn:
    #   "duration_min", "is_member", "revenue_est"
    # - "is_member" powinno mieć dtype bool
    # - "duration_min" i "revenue_est" powinny być nieujemne
    raise NotImplementedError


feat = add_trip_features(trips_small)
assert {"duration_min", "is_member", "revenue_est"}.issubset(feat.columns)
assert feat["duration_min"].ge(0).all()
assert feat["is_member"].dtype == bool
assert feat["revenue_est"].ge(0).all()
feat[["duration_min", "is_member", "revenue_est"]].head()


### Cwiczenie 3.03.2 - Alignment po indeksie: dzienne cechy popytu i pogody

Zbuduj tabele indeksowana po dacie z kolumnami:
- `n_trips`
- `mean_temp_c`
- `total_precip_mm`
- `trips_per_mm`

**Wymagane metody**
- ustawienie indeksu czasowego,
- `resample('D')`,
- polaczenie przez alignment indeksu (`join` albo `concat`), **bez `merge()`**,
- ochrona przed dzieleniem przez zero przy `trips_per_mm`.


In [ ]:
def daily_weather_alignment(trips: pd.DataFrame, weather: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Połącz dzienne statystyki przejazdów i pogody,
    # ale NIE używaj merge().
    #
    # Celem jest alignment po indeksie czasowym:
    # - dla trips policz liczbę przejazdów dziennie
    # - dla weather policz dzienne statystyki pogody
    # - potem połącz oba obiekty po wspólnym indeksie dat
    #
    # Wynik ma mieć DatetimeIndex oraz kolumny:
    # - "n_trips"
    # - "mean_temp_c"
    # - "total_precip_mm"
    # - "trips_per_mm"
    #
    # Krok 1. Dzienne przejazdy:
    # - ustaw indeks czasowy na podstawie kolumny "start_ts"
    # - użyj pd.to_datetime(...)
    # - następnie resample("D")
    # - policz liczbę wierszy w każdym dniu
    # - nazwij wynik "n_trips"
    #
    # Wskazówka:
    # - wygodny schemat to:
    #       trips.set_index(pd.to_datetime(trips["start_ts"]))
    # - potem:
    #       .resample("D").size()
    # - na końcu:
    #       .rename("n_trips")
    #
    # Krok 2. Dzienne statystyki pogody:
    # - ustaw indeks na kolumnie "timestamp_hour"
    # - upewnij się, że dane są posortowane po czasie
    # - przeskaluj do danych dziennych przez resample("D")
    # - policz:
    #       mean_temp_c     -> średnia z "temp_c"
    #       total_precip_mm -> suma z "precip_mm"
    #
    # Wskazówki:
    # - użyj:
    #       weather.set_index("timestamp_hour")
    # - potem:
    #       .sort_index()
    # - a następnie:
    #       .resample("D").agg(...)
    #
    # Krok 3. Alignment po indeksie:
    # - połącz daily_trips i daily_weather po indeksie czasu
    # - nie używaj merge()
    # - użyj narzędzia, które skleja obiekty według indeksu,
    #   np. pd.concat([...], axis=1)
    #
    # Krok 4. Dodatkowy wskaźnik:
    # - policz:
    #       trips_per_mm = n_trips / total_precip_mm
    #
    # Uwaga:
    # - gdy total_precip_mm == 0, uniknij dzielenia przez zero
    # - najprościej najpierw zamienić zera w total_precip_mm na NaN
    # - przyda się:
    #       .replace(0, np.nan)
    #
    # Krok 5. Zwróć wynik:
    # - wynik powinien mieć rosnący DatetimeIndex
    # - suma kolumny "n_trips" powinna dać liczbę wszystkich przejazdów
    raise NotImplementedError

aligned = daily_weather_alignment(trips_small, weather_small)
assert isinstance(aligned.index, pd.DatetimeIndex)
assert {"n_trips", "mean_temp_c", "total_precip_mm", "trips_per_mm"}.issubset(aligned.columns)
assert int(aligned["n_trips"].sum()) == len(trips_small)
assert aligned.index.is_monotonic_increasing
aligned.head()

## 03.04 - Missing Values


### Cwiczenie 3.04.1 - Uzupelnij braki w pogodzie

Uzupelnij:
- `temp_c` przez interpolacje po czasie,
- `precip_mm` wartoscia 0.

Zwracany DataFrame ma nie miec brakow w tych dwoch kolumnach.


In [ ]:
def fill_weather_missing(weather: pd.DataFrame) -> pd.DataFrame:
    # TODO pseudokod + wskazówki:
    #
    # 1) uporządkuj rekordy po czasie i pracuj na kopii
    #    podpowiedź:
    #    weather.sort_values("timestamp_hour").copy()
    #
    # 2) ustaw kolumnę czasu jako indeks
    #    podpowiedź:
    #    ... .set_index("timestamp_hour")
    #
    # 3) uzupełnij braki temperatury, korzystając z przebiegu w czasie
    #    podpowiedź:
    #    out["temp_c"] = out["temp_c"].interpolate(method="time")
    #
    # 4) jeśli po interpolacji zostaną braki na początku lub końcu,
    #    dopełnij je najbliższą znaną wartością
    #    podpowiedź:
    #    ... .ffill().bfill()
    #
    # 5) w opadzie przyjmij, że brak oznacza 0.0
    #    podpowiedź:
    #    out["precip_mm"] = out["precip_mm"].fillna(0.0)
    #
    # 6) przywróć zwykły układ tabeli
    #    podpowiedź:
    #    out.reset_index()
    #
    # 7) zwróć wynik
    raise NotImplementedError


weather_filled = fill_weather_missing(weather_small)
# assert isinstance(weather_filled, pd.DataFrame)
assert weather_filled["temp_c"].isna().sum() == 0
assert weather_filled["precip_mm"].isna().sum() == 0
weather_filled.head()


### Cwiczenie 3.04.2 - Audit brakow danych

Zbuduj tabele jakosci danych dla dowolnego DataFrame, zawierajaca kolumny:
- `variable`
- `n_missing`
- `pct_missing`
- `dtype`

Posortuj wynik malejaco po `n_missing`.

**Wymagane metody**
- `isna()`,
- agregacje po kolumnach,
- zbudowanie nowego `DataFrame` z podsumowaniem.


In [ ]:
def missing_audit(df: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # out = pd.DataFrame(
    #     {
    #         "variable": ...,
    #         "n_missing": ...,
    #         "pct_missing": ...,
    #         "dtype": ...,
    #     }
    # )
    #
    # out = out.sort_values(
    #     ["n_missing", "variable"],
    #     ascending=[False, True],
    #     ignore_index=True,
    # )
    #
    # return out
    raise NotImplementedError

audit = missing_audit(weather_small)
assert list(audit.columns) == ["variable", "n_missing", "pct_missing", "dtype"]
assert audit["n_missing"].is_monotonic_decreasing
assert set(audit["variable"]) == set(weather_small.columns)
audit

## 03.05 - Hierarchical Indexing


### Cwiczenie 3.05.1 - MultiIndex: popyt per dzielnica i godzina

Zbuduj `pd.Series` z indeksem wielopoziomowym `(district, hour)`,
ktora zawiera liczbe przejazdow startujacych z danej dzielnicy o danej godzinie.


In [ ]:
def demand_multiindex(trips: pd.DataFrame, stations: pd.DataFrame) -> pd.Series:
    # TODO:
    # Zbuduj serię popytu z indeksem wielopoziomowym (district, hour),
    # która zlicza liczbę przejazdów startujących w danej dzielnicy
    # i o danej godzinie.
    #
    # Wynik:
    # - ma być pd.Series
    # - ma mieć MultiIndex z 2 poziomami:
    #   1) district
    #   2) hour
    # - wartości mają oznaczać liczbę przejazdów
    # - suma wartości powinna dać liczbę wszystkich wierszy w trips
    #
    # TODO pseudokod + wskazówki:
    #
    # 1) dołącz do trips informację o district dla stacji startowej
    #    - w trips identyfikator stacji startowej to "true_start_station_id"
    #    - w stations identyfikator stacji to "station_id"
    #    - potrzebujesz tylko kolumn:
    #         "station_id", "district"
    #    - podpowiedź:
    #      trips.merge(..., left_on=..., right_on=..., how="left")
    #
    # 2) z kolumny "start_ts" wyciągnij godzinę startu
    #    - podpowiedź:
    #      pd.to_datetime(...).dt.hour
    #    - zapisz wynik do nowej kolumny "hour"
    #
    # 3) pogrupuj dane po:
    #    - "district"
    #    - "hour"
    #    i policz liczność każdej grupy
    #    - podpowiedź:
    #      .groupby(["district", "hour"]).size()
    #
    # 4) uporządkuj wynik po indeksie
    #    - podpowiedź:
    #      .sort_index()
    #
    # 5) nadaj serii nazwę, np.:
    #      out.name = "n_trips"
    #
    # 6) zwróć wynikową serię
    raise NotImplementedError


mi = demand_multiindex(trips_small, stations_small)
assert isinstance(mi, pd.Series)
assert mi.index.nlevels == 2
assert mi.sum() == len(trips_small)
mi.head(10)


### Cwiczenie 3.05.2 - MultiIndex: przekroje (`.xs`) i slicing godzin

Mamy juz serie `mi` z indeksem `(district, hour)`. Teraz przećwicz:

1. Pobieranie przekroju dla jednej dzielnicy (profil godzinowy).
2. Wycinanie zakresu godzin dla wszystkich dzielnic przez `pd.IndexSlice`.

Uwaga: slicing po poziomie `hour` wymaga, aby indeks byl posortowany (`sort_index()`).


In [ ]:
def district_profile_hours(mi: pd.Series, district: str) -> pd.Series:
    # TODO:
    # Z mi (Series z MultiIndex: district, hour) wybierz tylko jeden district
    # i zwróć profil godzinowy jako zwykłą Series indeksowaną po hour.
    #
    # PSEUDOKOD + wskazówki:
    # 1) wybierz przekrój po poziomie "district"
    #    podpowiedź:
    #    mi.xs(district, level="district")
    #
    # 2) uporządkuj wynik rosnąco po godzinie
    #    podpowiedź:
    #    .sort_index()
    #
    # 3) dopilnuj, aby indeks wynikowej serii miał nazwę "hour"
    #    podpowiedź:
    #    out.index.name = "hour"
    #
    # 4) zwróć wynik
    raise NotImplementedError


def slice_hours(mi: pd.Series, hour_start: int, hour_end: int) -> pd.Series:
    # TODO:
    # Z Series z MultiIndex (district, hour) wybierz tylko te rekordy,
    # których hour należy do przedziału [hour_start, hour_end].
    #
    # PSEUDOKOD + wskazówki:
    # 1) najpierw posortuj serię po indeksie
    #    podpowiedź:
    #    mi_sorted = mi.sort_index()
    #
    # 2) przygotuj wygodny "selector" do cięcia MultiIndex
    #    podpowiedź:
    #    idx = pd.IndexSlice
    #
    # 3) wybierz wszystkie district, ale tylko godziny z zadanego zakresu
    #    schemat:
    #    mi_sorted.loc[idx[:, hour_start:hour_end]]
    #
    # 4) zwróć wynik
    raise NotImplementedError


some_district = mi.index.get_level_values("district")[0]

prof = district_profile_hours(mi, some_district)
sub = slice_hours(mi, 7, 9)

assert isinstance(prof, pd.Series)
assert int(prof.sum()) == int(mi.xs(some_district, level="district").sum())
assert set(sub.index.get_level_values("hour")).issubset(set([7, 8, 9]))

prof.head(), sub.head()


### Cwiczenie 3.05.3 - MultiIndex: pelna siatka + `unstack` (heatmap)

W praktyce nie wszystkie kombinacje `(district, hour)` muszą występować w danych.
Zbuduj więc **pelna siatke** (district x 24 godziny), wypelnij braki zerami,
a nastepnie przejdz do postaci macierzy przez `unstack('hour')`.


In [ ]:
def demand_heatmap(mi: pd.Series, districts: list[str]) -> pd.DataFrame:
    # TODO:
    # Z Series z MultiIndex (district, hour) zbuduj tabelę 2D,
    # w której:
    # - wiersze to district
    # - kolumny to godziny 0..23
    # - wartości to liczby przejazdów
    #
    # Wynik ma mieć:
    # - tyle wierszy, ile jest district w liście districts
    # - dokładnie 24 kolumny: 0, 1, ..., 23
    # - brakujące kombinacje (district, hour) uzupełnione zerami
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zbuduj pełny MultiIndex wszystkich możliwych par:
    #    (district, hour), gdzie hour = 0..23
    #    podpowiedź:
    #    pd.MultiIndex.from_product(
    #        [districts, range(24)],
    #        names=["district", "hour"]
    #    )
    #
    # 2) dopasuj mi do tego pełnego indeksu
    #    tak, aby brakujące pary dostały wartość 0
    #    podpowiedź:
    #    mi.reindex(full_idx, fill_value=0)
    #
    # 3) jeśli chcesz, zamień wartości na int
    #    podpowiedź:
    #    .astype(int)
    #
    # 4) uporządkuj serię po MultiIndex
    #    podpowiedź:
    #    .sort_index()
    #
    # 5) zamień poziom "hour" z indeksu na kolumny
    #    tak, aby dostać DataFrame
    #    podpowiedź:
    #    .unstack("hour")
    #
    # 6) uporządkuj:
    #    - wiersze po district
    #    - kolumny po hour
    #    podpowiedzi:
    #    .sort_index()
    #    .sort_index(axis=1)
    #
    # 7) zwróć gotowy DataFrame
    raise NotImplementedError


districts = sorted(stations_small["district"].unique().tolist())
heat = demand_heatmap(mi, districts)

assert isinstance(heat, pd.DataFrame)
assert heat.shape == (len(districts), 24)
assert int(heat.to_numpy().sum()) == len(trips_small)

heat.head()


## 03.06 - Concat and Append


### Cwiczenie 3.06.1 - Polacz tygodniowe paczki

Podziel `trips_small` na 3 czesci i napisz funkcje, ktora zlacza je z powrotem
tak, aby zachowac wszystkie wiersze i zresetowac indeks.


In [ ]:

parts = [
    trips_small.iloc[:1500].copy(),
    trips_small.iloc[1500:3200].copy(),
    trips_small.iloc[3200:].copy(),
]

def concat_trip_parts(parts: list[pd.DataFrame]) -> pd.DataFrame:
    # TODO:
    # Połącz listę fragmentów DataFrame w jeden wspólny DataFrame.
    #
    # Wymagania:
    # 1) użyj pd.concat(...)
    # 2) zachowaj wszystkie wiersze ze wszystkich elementów listy parts
    # 3) ustaw nowy, ciągły indeks 0, 1, 2, ...
    #
    # Wskazówka:
    # - przyda się argument:
    #       ignore_index=True
    #
    # Schemat:
    # joined = pd.concat(parts, ignore_index=True)
    # return joined
    raise NotImplementedError


joined = concat_trip_parts(parts)
assert isinstance(joined, pd.DataFrame)
assert len(joined) == len(trips_small)
assert joined.index.equals(pd.RangeIndex(len(joined)))
joined.head()


### Cwiczenie 3.06.2 - `concat` z kluczami: zachowaj pochodzenie partii

Polacz te same paczki `parts`, ale tym razem zachowaj informacje o zrodlowej partii
w zewnetrznym poziomie indeksu.

Wynik ma miec `MultiIndex` o poziomach:
- `batch`
- `row_id`

**Wymagane metody**
- `pd.concat(..., keys=..., names=...)`


In [ ]:
def concat_trip_parts_with_keys(parts: list[pd.DataFrame]) -> pd.DataFrame:
    # TODO:
    # Połącz części DataFrame w jeden obiekt, ale tym razem zachowaj informację,
    # z której części pochodzi każdy wiersz.
    #
    # Wynik:
    # - ma mieć MultiIndex
    # - pierwszy poziom indeksu ma opisywać batch / część
    # - drugi poziom ma odpowiadać oryginalnemu indeksowi w danej części
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) przygotuj listę etykiet dla kolejnych części
    #    np.:
    #    "part_1", "part_2", ..., "part_n"
    #
    #    podpowiedź:
    #    [f"part_{i + 1}" for i in range(len(parts))]
    #
    # 2) użyj pd.concat(...), przekazując:
    #    - parts jako listę DataFrame
    #    - keys=...  -> etykiety części
    #    - names=... -> nazwy poziomów indeksu
    #
    #    podpowiedź:
    #    pd.concat(parts, keys=keys, names=["batch", "row_id"])
    #
    # 3) zwróć wynik
    raise NotImplementedError


joined_keys = concat_trip_parts_with_keys(parts)
assert isinstance(joined_keys.index, pd.MultiIndex)
assert joined_keys.index.names == ["batch", "row_id"]
assert joined_keys.index.get_level_values("batch").nunique() == len(parts)
assert len(joined_keys) == len(trips_small)
joined_keys.head()

## 03.07 - Merge and Join


### Checkpoint 3.07.A — zanim zrobisz `merge`, nazwij relację

To jest krótki, ale bardzo ważny nawyk analityczny.

Zanim napiszesz `merge(...)`, odpowiedz:
- jaka jest relacja między `trips` i `stations`?
- jaka jest relacja między `trips` i pogodą po zaokrągleniu czasu do godziny?
- dlaczego tutaj sensowny jest `left join`, a nie `inner join`?

Nie chodzi o teorię dla teorii — chodzi o to, żeby po scaleniu nie zgubić ani nie rozmnożyć wierszy.


In [ ]:
CARDINALITY_STATIONS = ""   # TODO:
# wpisz typ relacji między trips a stations
# z perspektywy tabeli trips:
# - wiele przejazdów może wskazywać na jedną stację
# - więc oczekiwany zapis to np. "many_to_one"

CARDINALITY_WEATHER = ""    # TODO:
# wpisz typ relacji między trips a weather
# z perspektywy tabeli trips:
# - wiele przejazdów może przypadać na ten sam rekord pogody
#   (np. po dopasowaniu do tej samej godziny / dnia)
# - więc oczekiwany zapis to np. "many_to_one"
WHY_LEFT_JOIN = (
    "Tabela `trips` jest tutaj tabelą główną i każda obserwacja opisuje pojedynczy przejazd. "
    "Do jednej stacji może odnosić się wiele przejazdów, a po zaokrągleniu czasu do godziny "
    "wiele przejazdów może wskazywać ten sam rekord pogody, więc w obu przypadkach relacja ma "
    "charakter `many_to_one`. "
    "Używamy `left join`, bo chcemy zachować pełną liczbę przejazdów nawet wtedy, gdy metadane stacji "
    "albo pogody okażą się niekompletne; `inner join` mógłby cicho wyrzucić część danych."
)         # TODO: 2-4 zdania

assert "many_to_one" in CARDINALITY_STATIONS.replace("-", "_").lower()
assert "many_to_one" in CARDINALITY_WEATHER.replace("-", "_").lower()
assert isinstance(WHY_LEFT_JOIN, str) and len(WHY_LEFT_JOIN.strip()) >= 100
print("Checkpoint 3.07.A ✅")


### Cwiczenie 3.07.1 - Polacz przejazdy, stacje i pogode

Zbuduj DataFrame z:
- przejazdami,
- metadanymi stacji startowej (`station_name`, `district`, `capacity`),
- pogoda dla godziny startu (`temp_c`, `precip_mm`, `wind_kmh`).

Wskazowka: pogode dolacz po `start_ts.dt.floor('h')`.


In [ ]:
def merge_trip_context(
    trips: pd.DataFrame,
    stations: pd.DataFrame,
    weather: pd.DataFrame,
) -> pd.DataFrame:
    # TODO:
    # Zbuduj tabelę przejazdów wzbogaconą o:
    # - metadane stacji startowej
    # - dane pogodowe dopasowane do godziny startu przejazdu
    #
    # Wynik powinien zachować:
    # - wszystkie wiersze z trips
    # - kolumny opisujące stację, np. station_name, district, capacity
    # - kolumny pogodowe, np. temp_c, precip_mm, wind_kmh
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii tabeli trips
    #    podpowiedź:
    #    out = trips.copy()
    #
    # 2) utwórz pomocniczą kolumnę z godziną startu przejazdu
    #    - weź "start_ts"
    #    - zamień na datetime
    #    - obetnij czas do pełnej godziny
    #    - zapisz np. jako "start_hour_ts"
    #
    #    podpowiedź:
    #    pd.to_datetime(out["start_ts"]).dt.floor("h")
    #
    # 3) przygotuj węższą tabelę stacji tylko z potrzebnymi kolumnami
    #    np.:
    #    - "station_id"
    #    - "station_name"
    #    - "district"
    #    - "capacity"
    #
    #    podpowiedź:
    #    stations[[...]].copy()
    #
    # 4) dołącz metadane stacji do out
    #    - klucz w trips: "true_start_station_id"
    #    - klucz w stations: "station_id"
    #    - użyj how="left", aby nie zgubić przejazdów
    #
    #    podpowiedź:
    #    out.merge(..., left_on=..., right_on=..., how="left")
    #
    # 5) dołącz dane pogodowe do out
    #    - klucz w out: "start_hour_ts"
    #    - klucz w weather: "timestamp_hour"
    #    - również użyj how="left"
    #
    # 6) po merge ze stacjami może zostać pomocnicza kolumna "station_id"
    #    - jeśli nie jest już potrzebna, usuń ją
    #    podpowiedź:
    #    .drop(columns=[...])
    #
    # 7) zwróć wynik
    raise NotImplementedError


merged = merge_trip_context(trips_small, stations_small, weather_filled)
required = {"station_name", "district", "capacity", "temp_c", "precip_mm", "wind_kmh"}
assert required.issubset(merged.columns)
assert len(merged) == len(trips_small)
merged.head()


### Cwiczenie 3.07.2 - Merge audit przez `validate` i `indicator`

Zdarza sie, ze w logach pojawia sie bledny `station_id`. Stworz kopie `trips`,
ustaw w pierwszym wierszu `true_start_station_id = -999`, a nastepnie wykonaj merge audytowy z tabela stacji.

Wynik ma:
- zachowac wszystkie wiersze (`how='left'`),
- sprawdzic relacje `many_to_one`,
- dodac kolumne pochodzenia przez `indicator=True`.

Zwracaj:
1. scalony `DataFrame`,
2. `Series` z liczebnoscia `_merge`.

**Wymagane metody**
- `merge`,
- `validate='many_to_one'`,
- `indicator=True`.


In [ ]:
def audited_station_merge(trips: pd.DataFrame, stations: pd.DataFrame):
    # TODO:
    # Wykonaj kontrolowany merge trips ze stacjami i zwróć:
    # 1) pełny DataFrame po merge
    # 2) krótkie podsumowanie jakości merge w postaci liczności wartości z kolumny "_merge"
    #
    # Celem jest:
    # - zachować wszystkie wiersze z trips
    # - sprawdzić oczekiwaną kardynalność relacji
    # - dostać informację, które rekordy dopasowały się poprawnie,
    #   a które pozostały bez pary
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) przygotuj węższą tabelę stacji tylko z potrzebnymi kolumnami
    #    podpowiedź:
    #    stations[["station_id", "station_name", "district", "capacity"]].copy()
    #
    # 2) wykonaj merge:
    #    - lewa tabela: trips
    #    - prawa tabela: stations_meta
    #    - klucz z lewej: "true_start_station_id"
    #    - klucz z prawej: "station_id"
    #    - użyj how="left"
    #
    # 3) dodaj kontrolę kardynalności
    #    - oczekujemy relacji many-to-one
    #    - podpowiedź:
    #      validate="many_to_one"
    #
    # 4) poproś Pandas o kolumnę diagnostyczną pokazującą źródło dopasowania
    #    - podpowiedź:
    #      indicator=True
    #
    # 5) po merge policz liczności kategorii z kolumny "_merge"
    #    - interesują nas m.in.:
    #      "both", "left_only", "right_only"
    #    - podpowiedź:
    #      out["_merge"].value_counts()
    #
    # 6) dla czytelności uporządkuj wynikowe counts po indeksie
    #    podpowiedź:
    #    .sort_index()
    #
    # 7) zwróć:
    #    - DataFrame po merge
    #    - Series z counts
    raise NotImplementedError


trips_probe = trips_small.copy()
trips_probe.loc[0, "true_start_station_id"] = -999

merge_audit_df, merge_audit_counts = audited_station_merge(trips_probe, stations_small)
assert "_merge" in merge_audit_df.columns
assert merge_audit_counts.index.isin(["left_only", "right_only", "both"]).all()
assert int(merge_audit_counts.get("left_only", 0)) >= 1
assert len(merge_audit_df) == len(trips_probe)
merge_audit_counts, merge_audit_df.head()

## 03.08 - Aggregation and Grouping


### Cwiczenie 3.08.1 - Dzienne KPI stacji

Dla kazdej pary `(date, true_start_station_id)` policz:
- `n_trips`
- `mean_duration_min`
- `total_revenue`

Wynik powinien byc DataFrame z kolumnami o takich nazwach.


In [ ]:
def station_daily_kpis(merged: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Zbuduj dzienne KPI dla każdej stacji startowej.
    #
    # Wynik ma zawierać co najmniej kolumny:
    # - "date"
    # - "true_start_station_id"
    # - "n_trips"
    # - "mean_duration_min"
    # - "total_revenue"
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii danych
    #    podpowiedź:
    #    out = merged.copy()
    #
    # 2) upewnij się, że masz kolumnę z samą datą przejazdu
    #    - weź "start_ts"
    #    - zamień na datetime
    #    - obetnij do początku dnia
    #    - zapisz do kolumny "date"
    #
    #    podpowiedź:
    #    pd.to_datetime(out["start_ts"]).dt.floor("D")
    #
    # 3) pogrupuj dane po:
    #    - "date"
    #    - "true_start_station_id"
    #
    #    podpowiedź:
    #    out.groupby(["date", "true_start_station_id"], as_index=False)
    #
    # 4) w agg(...) policz KPI:
    #    - n_trips            -> liczba przejazdów w grupie
    #    - mean_duration_min  -> średni czas przejazdu
    #    - total_revenue      -> suma revenue_est
    #
    #    podpowiedź:
    #    .agg(
    #        n_trips=("trip_id", "size"),
    #        mean_duration_min=("duration_min", "mean"),
    #        total_revenue=("revenue_est", "sum"),
    #    )
    #
    # 5) uporządkuj wynik po:
    #    - "date"
    #    - "true_start_station_id"
    #
    #    podpowiedź:
    #    .sort_values(["date", "true_start_station_id"])
    #
    # 6) zwróć wynikowy DataFrame
    raise NotImplementedError


merged_feat = add_trip_features(merged)
kpis = station_daily_kpis(merged_feat)
assert isinstance(kpis, pd.DataFrame)
assert {"date", "true_start_station_id", "n_trips", "mean_duration_min", "total_revenue"}.issubset(kpis.columns)
assert kpis["n_trips"].ge(1).all()
kpis.head()


### Cwiczenie 3.08.2 - `transform` i `filter` po grupach

Na bazie `merged_feat`:
- dodaj `district_median_duration`,
- dodaj `duration_vs_district_median`,
- zostaw tylko te dzielnice, ktore maja co najmniej `min_trips` przejazdow.

**Wymagane metody**
- `groupby(...).transform(...)`,
- `groupby(...).filter(...)`.

Nie uzywaj `merge`-back z osobna tabelka agregatow.


In [ ]:
def duration_vs_district_profile(merged_feat: pd.DataFrame, min_trips: int = 500) -> pd.DataFrame:
    # TODO:
    # Dla każdego przejazdu policz, jak jego czas trwania wypada
    # względem mediany czasu trwania w jego district.
    #
    # Wynik ma:
    # - zachować obserwacje wiersz po wierszu
    # - dodać kolumnę z medianą district
    # - dodać kolumnę z odchyleniem od tej mediany
    # - odrzucić district, które mają zbyt mało przejazdów
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii danych
    #    podpowiedź:
    #    out = merged_feat.copy()
    #
    # 2) dla każdej dzielnicy policz medianę "duration_min",
    #    ale wynik przypisz z powrotem do każdego wiersza tej grupy
    #    - tutaj nie używamy agg, tylko transform
    #
    #    podpowiedź:
    #    out.groupby("district")["duration_min"].transform("median")
    #
    #    zapisz wynik do kolumny:
    #    "district_median_duration"
    #
    # 3) policz, o ile dany przejazd różni się od mediany swojej dzielnicy
    #    schemat:
    #    duration_min - district_median_duration
    #
    #    zapisz wynik do kolumny:
    #    "duration_vs_district_median"
    #
    # 4) odrzuć te district, które mają mniej niż min_trips obserwacji
    #    - tutaj przyda się groupby(...).filter(...)
    #    - funkcja w filter powinna zostawić tylko grupy o długości >= min_trips
    #
    #    podpowiedź:
    #    out.groupby("district").filter(lambda g: len(g) >= min_trips)
    #
    # 5) po filter dobrze zrobić .copy()
    #
    # 6) zwróć wynik
    raise NotImplementedError


district_prof = duration_vs_district_profile(merged_feat, min_trips=500)
assert {"district_median_duration", "duration_vs_district_median"}.issubset(district_prof.columns)
assert district_prof.groupby("district").size().ge(500).all()
assert np.allclose(
    district_prof["duration_vs_district_median"],
    district_prof["duration_min"] - district_prof["district_median_duration"],
)
district_prof.head()

## 03.09 - Pivot Tables


### Cwiczenie 3.09.1 - Macierz: weekday x hour

Zbuduj pivot table z liczba przejazdow:
- wiersze: `weekday`
- kolumny: `hour`
- wartosci: liczba przejazdow

Ustal kolejnosc dni: `Mon, Tue, Wed, Thu, Fri, Sat, Sun`.


In [ ]:
def weekday_hour_pivot(trips: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Zbuduj tabelę przestawną pokazującą liczbę przejazdów
    # w układzie:
    # - wiersze: dzień tygodnia
    # - kolumny: godzina 0..23
    #
    # Wynik:
    # - ma być DataFrame
    # - suma wszystkich komórek ma dawać liczbę przejazdów
    # - brakujące kombinacje weekday/hour mają być uzupełnione zerami
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii danych
    #    podpowiedź:
    #    out = trips.copy()
    #
    # 2) z kolumny "start_ts" wyznacz nazwę dnia tygodnia
    #    - możesz użyć day_name()
    #    - potem skróć nazwę do 3 liter, np. Mon, Tue, Wed
    #
    #    podpowiedź:
    #    pd.to_datetime(out["start_ts"]).dt.day_name().str[:3]
    #
    # 3) ustaw "weekday" jako kategorię uporządkowaną,
    #    aby dni były w poprawnej kolejności:
    #    Mon, Tue, Wed, Thu, Fri, Sat, Sun
    #
    #    podpowiedź:
    #    pd.Categorical(
    #        ...,
    #        categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    #        ordered=True,
    #    )
    #
    # 4) z "start_ts" wyciągnij godzinę startu
    #    podpowiedź:
    #    pd.to_datetime(out["start_ts"]).dt.hour
    #
    #    zapisz do kolumny "hour"
    #
    # 5) zbuduj pivot_table:
    #    - index="weekday"
    #    - columns="hour"
    #    - jako zliczanie użyj np. "trip_id"
    #    - aggfunc="size"
    #    - fill_value=0
    #
    #    podpowiedź:
    #    out.pivot_table(...)
    #
    # 6) upewnij się, że wynik zawiera wszystkie dni tygodnia
    #    nawet jeśli w danych jakiegoś dnia nie było
    #    podpowiedź:
    #    .reindex(index=[...], fill_value=0)
    #
    # 7) upewnij się, że wynik zawiera wszystkie godziny 0..23
    #    podpowiedź:
    #    .reindex(columns=range(24), fill_value=0)
    #
    # 8) zwróć wynik
    raise NotImplementedError


pivot = weekday_hour_pivot(trips_small)
assert isinstance(pivot, pd.DataFrame)
assert pivot.shape[1] <= 24
assert pivot.to_numpy().sum() == len(trips_small)
pivot


### Cwiczenie 3.09.2 - Pivot wielopoziomowy: `fill_value` i `margins`

Zbuduj pivot z przychodem szacowanym (`revenue_est`):
- `index=['district', 'user_type']`
- `columns='weekday'`
- `values='revenue_est'`
- `aggfunc=['sum', 'mean']`
- `fill_value=0`
- `margins=True`

**Wymagane metody**
- `pivot_table`,
- `fill_value`,
- `margins`.


In [ ]:
def revenue_pivot_multi(merged_feat: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Zbuduj wielopoziomową tabelę przestawną dla revenue_est.
    #
    # Cel:
    # - wiersze: (district, user_type)
    # - kolumny: weekday
    # - wartości: revenue_est
    # - agregacje: suma i średnia
    # - dodatkowo dodaj marginesy "All"
    #
    # Wynik powinien mieć:
    # - MultiIndex w wierszach
    # - MultiIndex w kolumnach
    # - m.in. kolumnę ("sum", "All")
    # - m.in. wiersz ("All", "")
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii danych
    #    podpowiedź:
    #    out = merged_feat.copy()
    #
    # 2) przygotuj kolumnę "weekday" na podstawie "start_ts"
    #    - wyciągnij nazwę dnia tygodnia
    #    - skróć do 3 liter
    #    - ustaw jako kategorię uporządkowaną:
    #      Mon, Tue, Wed, Thu, Fri, Sat, Sun
    #
    #    podpowiedź:
    #    pd.Categorical(
    #        pd.to_datetime(out["start_ts"]).dt.day_name().str[:3],
    #        categories=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    #        ordered=True,
    #    )
    #
    # 3) zbuduj pivot_table dla "revenue_est"
    #    - index=["district", "user_type"]
    #    - columns="weekday"
    #    - values="revenue_est"
    #    - aggfunc=["sum", "mean"]
    #    - fill_value=0
    #    - margins=True   -> żeby dostać "All"
    #
    #    podpowiedź:
    #    out.pivot_table(...)
    #
    # 4) po pivot_table uporządkuj kolumny
    #    tak, aby dla obu agregacji kolejność dni była:
    #    Mon, Tue, Wed, Thu, Fri, Sat, Sun, All
    #
    # 5) w tym celu możesz jawnie zbudować docelowy MultiIndex kolumn
    #    podpowiedź:
    #    pd.MultiIndex.from_tuples(
    #        [(agg, day) for agg in ["sum", "mean"]
    #                    for day in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun", "All"]],
    #        names=table.columns.names,
    #    )
    #
    # 6) przereindexuj kolumny do tego docelowego układu
    #    i ewentualne braki wypełnij zerami
    #
    #    podpowiedź:
    #    table.reindex(columns=desired_cols).fillna(0)
    #
    # 7) zwróć wynik
    raise NotImplementedError


rev_pivot = revenue_pivot_multi(merged_feat)
assert isinstance(rev_pivot.index, pd.MultiIndex)
assert isinstance(rev_pivot.columns, pd.MultiIndex)
assert ("sum", "All") in rev_pivot.columns
assert ("All", "") in rev_pivot.index
rev_pivot.head()

## 03.10 - Working With Strings


### Cwiczenie 3.10.1 - Oczysc nazwy stacji i kupony

Napisz funkcje, ktora:
- tworzy kolumne `station_name_clean` w `stations`,
- tworzy kolumne `coupon_clean` w `trips`,
- obcina spacje, zamienia litery na male, a brakujace kupony wpisuje jako `"none"`.


In [ ]:
def clean_text_columns(trips: pd.DataFrame, stations: pd.DataFrame):
    # TODO:
    # Przygotuj "oczyszczone" wersje wybranych kolumn tekstowych.
    #
    # Masz zwrócić dwie kopie danych:
    # - trips2
    # - stations2
    #
    # I dodać do nich nowe kolumny:
    # - w trips2:    "coupon_clean"
    # - w stations2: "station_name_clean"
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii obu DataFrame
    #    podpowiedź:
    #    trips2 = trips.copy()
    #    stations2 = stations.copy()
    #
    # 2) oczyść kolumnę "coupon_code" i zapisz wynik do "coupon_clean"
    #    Chcemy:
    #    - brakujące wartości zastąpić napisem "none"
    #    - zamienić dane na tekst
    #    - usunąć spacje z początku i końca
    #    - zamienić litery na małe
    #    - jeśli po czyszczeniu wyjdzie pusty napis "", też zamień go na "none"
    #
    #    podpowiedzi:
    #    .fillna("none")
    #    .astype(str)
    #    .str.strip()
    #    .str.lower()
    #    .replace("", "none")
    #
    # 3) oczyść kolumnę "station_name" i zapisz wynik do "station_name_clean"
    #    Chcemy:
    #    - zamienić dane na tekst
    #    - usunąć spacje z początku i końca
    #    - zamienić litery na małe
    #
    #    podpowiedzi:
    #    .astype(str)
    #    .str.strip()
    #    .str.lower()
    #
    # 4) zwróć obie oczyszczone tabele:
    #    return trips2, stations2
    raise NotImplementedError

trips_txt, stations_txt = clean_text_columns(trips_small, stations_small)
assert "coupon_clean" in trips_txt.columns
assert "station_name_clean" in stations_txt.columns
assert trips_txt["coupon_clean"].isna().sum() == 0
trips_txt[["coupon_code", "coupon_clean"]].head(), stations_txt[["station_name", "station_name_clean"]].head()


### Cwiczenie 3.10.2 ⭐ - Regex: wyciagnij kod i dzielnice z nazwy stacji

Nazwy stacji maja trzy formaty:
- `st_00 | praga`
- `ST-01 / MOKOTOW`
- `Station 02 - Wola`

Wyciagnij z `station_name`:
- `station_code` (int),
- `district_from_name` (lowercase),
- `name_pattern` (`'pipe'`, `'slash'`, `'dash'`).

**Wymagane metody**
- `.str.extract(...)`,
- `.str.contains(...)`,
- `.str.lower()`.


In [ ]:
def parse_station_name_tokens(stations: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Z kolumny "station_name" wyciągnij kilka uporządkowanych informacji:
    # - "station_code"         -> numer stacji zapisany w nazwie
    # - "district_from_name"   -> dzielnica odczytana z końca nazwy
    # - "name_pattern"         -> typ separatora użytego w nazwie
    #
    # Wynik ma być kopią stations z dodatkowymi kolumnami.
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zacznij od kopii danych
    #    podpowiedź:
    #    stations2 = stations.copy()
    #
    # 2) z kolumny "station_name" wyciągnij 2 elementy regexem:
    #    - numer stacji (np. 02, 15, 21)
    #    - nazwę dzielnicy na końcu napisu
    #
    #    Wskazówka:
    #    - użyj .str.extract(...)
    #    - regex powinien mieć DWIE grupy przechwytujące:
    #      (1) numer stacji
    #      (2) district
    #
    #    Przydatny schemat:
    #    - początek z "St" albo "Station"
    #    - potem separator / spacje / "_" / "-"
    #    - potem dwucyfrowy kod
    #    - gdzieś dalej tekst
    #    - na końcu jedno słowo z nazwą dzielnicy
    #
    #    podpowiedź:
    #    stations2["station_name"].str.extract(r"...")
    #
    # 3) z wyniku extract:
    #    - pierwszą kolumnę zamień na int i zapisz do "station_code"
    #    - drugą kolumnę zamień na małe litery i zapisz do "district_from_name"
    #
    #    podpowiedzi:
    #    extracted[0].astype(int)
    #    extracted[1].str.lower()
    #
    # 4) utwórz kolumnę "name_pattern", która mówi,
    #    jaki separator pojawia się w nazwie stacji:
    #    - jeśli nazwa zawiera "|"  -> "pipe"
    #    - jeśli nazwa zawiera "/"  -> "slash"
    #    - w przeciwnym razie       -> "dash"
    #
    #    Wskazówka:
    #    - możesz użyć np.select(...)
    #    - warunki zbuduj przez .str.contains(..., regex=True)
    #
    #    podpowiedź:
    #    np.select(
    #        [warunek_pipe, warunek_slash],
    #        ["pipe", "slash"],
    #        default="dash",
    #    )
    #
    # 5) zwróć stations2
    raise NotImplementedError


stations_parsed = parse_station_name_tokens(stations_small)
assert {"station_code", "district_from_name", "name_pattern"}.issubset(stations_parsed.columns)
assert stations_parsed["station_code"].dtype.kind in "iu"
assert stations_parsed["district_from_name"].equals(stations_parsed["district"].str.lower())
assert set(stations_parsed["name_pattern"].unique()).issubset({"pipe", "slash", "dash"})
stations_parsed[["station_name", "station_code", "district_from_name", "name_pattern"]].head(8)

## 03.11 - Working with Time Series


### Cwiczenie 3.11.1 - Dzienna liczba przejazdow i srednia 7D

Zbuduj DataFrame indeksowany po dacie z kolumnami:
- `n_trips`
- `rolling_7d`

Wskazowka: `resample('D')`.


In [ ]:
def daily_counts_with_rolling(trips: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Zbuduj dzienny szereg liczby przejazdów i dodaj do niego
    # 7-dniową średnią kroczącą.
    #
    # Wynik ma:
    # - mieć DatetimeIndex
    # - zawierać kolumnę "n_trips"
    # - zawierać kolumnę "rolling_7d"
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) ustaw indeks czasowy na podstawie "start_ts"
    #    podpowiedź:
    #    pd.to_datetime(trips["start_ts"])
    #
    #    schemat:
    #    trips.set_index(...)
    #
    # 2) przeskaluj dane do częstości dziennej
    #    i policz liczbę przejazdów w każdym dniu
    #    podpowiedzi:
    #    .resample("D")
    #    .size()
    #
    # 3) nazwij tę serię jako "n_trips"
    #    podpowiedź:
    #    .rename("n_trips")
    #
    # 4) zamień serię na DataFrame
    #    podpowiedź:
    #    .to_frame()
    #
    # 5) dodaj kolumnę "rolling_7d"
    #    jako 7-dniową średnią kroczącą z "n_trips"
    #
    #    podpowiedź:
    #    daily["n_trips"].rolling(7, min_periods=1).mean()
    #
    #    Uwaga:
    #    - min_periods=1 sprawia, że pierwsze dni też dostaną wartość,
    #      nawet jeśli nie ma jeszcze pełnych 7 obserwacji
    #
    # 6) zwróć wynik
    raise NotImplementedError

daily = daily_counts_with_rolling(trips_small)
assert isinstance(daily.index, pd.DatetimeIndex)
assert {"n_trips", "rolling_7d"}.issubset(daily.columns)
assert daily["n_trips"].sum() == len(trips_small)
daily.head(10)


### Mini-case końcowy — napisz 3 krótkie obserwacje z danych (zrobione w zasadzie, ale możecie sobie rzucić okiem)

To jest mały most do projektu końcowego.

Na podstawie obiektów, które już policzyłeś/aś (`pivot`, `kpis`, `daily`, `merged_feat`),
napisz trzy krótkie obserwacje:
1. kiedy ruch wygląda na największy,
2. czy bardziej widać rytm godzinowy czy różnice między dniami,
3. która tabela / transformacja była do tego najwygodniejsza i dlaczego.

Nie chodzi o „jedyną poprawną odpowiedź”, tylko o to, żeby student ćwiczył
**przechodzenie od tabeli do wniosku**.


In [ ]:
OBSERVATION_1 = (
    "Największy ruch widać w godzinach popołudniowych, szczególnie około 17:00–18:00, "
    "a drugi wyraźny pik pojawia się rano około 7:00–8:00."
)
OBSERVATION_2 = (
    "Silniejszy jest rytm godzinowy niż różnice między dniami tygodnia: profile godzin mają "
    "bardzo wyraźne piki, podczas gdy sumy dzienne są do siebie dużo bardziej zbliżone."
)
METHOD_NOTE = (
    "Najwygodniejsza była tabela `pivot` po `(weekday, hour)`, bo w jednym miejscu pokazuje oba "
    "wymiary naraz i pozwala od razu porównać kształt profilu w ciągu dnia z różnicami między dniami. "
    "Uzupełniająco przydały się `daily` i `rolling_7d`, bo dobrze streszczają zmiany dzień po dniu, "
    "ale to właśnie pivot najłatwiej zamienia surowe logi przejazdów w krótki wniosek analityczny."
)

assert isinstance(OBSERVATION_1, str) and len(OBSERVATION_1.strip()) >= 40
assert isinstance(OBSERVATION_2, str) and len(OBSERVATION_2.strip()) >= 40
assert isinstance(METHOD_NOTE, str) and len(METHOD_NOTE.strip()) >= 100
print("Mini-case analityczny ✅")


### Cwiczenie 3.11.2 ⭐ - Szereg godzinowy: `shift` i porownanie do poprzedniej doby

Zbuduj godzinowy szereg popytu z kolumnami:
- `n_trips`
- `lag_24h`
- `rolling_24h_mean`
- `vs_prev_day`

gdzie `vs_prev_day = n_trips / lag_24h`.

**Wymagane metody**
- indeks czasowy,
- `resample('h')`,
- `shift(24)`,
- `rolling(24)`.


In [ ]:
def hourly_demand_lag_features(trips: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Zbuduj godzinowy szereg liczby przejazdów i dodaj cechy opóźnione
    # oraz cechy oparte na oknie kroczącym.
    #
    # Wynik ma:
    # - mieć DatetimeIndex
    # - zawierać kolumny:
    #   "n_trips"
    #   "lag_24h"
    #   "rolling_24h_mean"
    #   "vs_prev_day"
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) ustaw indeks czasowy na podstawie "start_ts"
    #    podpowiedź:
    #    pd.to_datetime(trips["start_ts"])
    #
    # 2) przeskaluj dane do częstości godzinowej
    #    i policz liczbę przejazdów w każdej godzinie
    #
    #    podpowiedzi:
    #    .set_index(...)
    #    .resample("h")
    #    .size()
    #    .rename("n_trips")
    #    .to_frame()
    #
    # 3) dodaj kolumnę "lag_24h"
    #    - ma oznaczać wartość n_trips sprzed dokładnie 24 godzin
    #    - pierwsze 24 rekordy powinny dać NaN
    #
    #    podpowiedź:
    #    hourly["n_trips"].shift(24)
    #
    # 4) dodaj kolumnę "rolling_24h_mean"
    #    - ma być średnią kroczącą z ostatnich 24 godzin
    #    - dla początku szeregu użyj min_periods=1
    #
    #    podpowiedź:
    #    hourly["n_trips"].rolling(24, min_periods=1).mean()
    #
    # 5) dodaj kolumnę "vs_prev_day"
    #    - ma porównywać bieżącą liczbę przejazdów z tą samą godziną
    #      dnia poprzedniego
    #    - schemat:
    #      n_trips / lag_24h
    #
    #    Uwaga:
    #    - jeśli lag_24h == 0, uniknij dzielenia przez zero
    #    - najpierw zamień zera w lag_24h na NaN
    #
    #    podpowiedź:
    #    hourly["lag_24h"].replace(0, np.nan)
    #
    # 6) zwróć wynik
    raise NotImplementedError


hourly = hourly_demand_lag_features(trips_small)
assert isinstance(hourly.index, pd.DatetimeIndex)
assert {"n_trips", "lag_24h", "rolling_24h_mean", "vs_prev_day"}.issubset(hourly.columns)
assert hourly["n_trips"].sum() == len(trips_small)
assert hourly["lag_24h"].iloc[:24].isna().all()
hourly.loc["2024-05-03"].head()

## 03.12 - Performance: eval() i query() (rozszerzenie praktyczne)


### Cwiczenie 3.12.1 - Szybki filtr po wielu warunkach

Na bazie `merged_feat` odfiltruj przejazdy, ktore spelniaja jednoczesnie:
- `duration_min >= 10`
- `duration_min <= 45`
- `precip_mm <= 1.0`
- `user_type == 'member'`

Zaimplementuj to przez `query()` lub `eval()`.


In [ ]:
def fast_member_filter(merged_feat: pd.DataFrame) -> pd.DataFrame:
    # TODO:
    # Odfiltruj tylko te przejazdy, które spełniają jednocześnie warunki:
    # - użytkownik jest typu "member"
    # - duration_min jest w przedziale od 10 do 45 minut
    # - precip_mm <= 1.0
    #
    # W tym zadaniu spróbuj użyć:
    # - query(...)
    # ewentualnie pomocniczo także eval(...), jeśli chcesz
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zbuduj warunek filtrowania w składni tekstowej dla query
    #    warunki:
    #    - 10 <= duration_min <= 45
    #    - precip_mm <= 1.0
    #    - user_type == 'member'
    #
    # 2) użyj:
    #    merged_feat.query("...")
    #
    # 3) zwróć kopię wyniku
    #    podpowiedź:
    #    .copy()
    #
    # Uwaga:
    # - wszystkie warunki mają być spełnione jednocześnie
    # - wynik ma pozostać DataFrame
    raise NotImplementedError
    

flt = fast_member_filter(merged_feat)
assert isinstance(flt, pd.DataFrame)
assert (flt["duration_min"] >= 10).all()
assert (flt["duration_min"] <= 45).all()
assert (flt["precip_mm"] <= 1.0).all()
assert (flt["user_type"] == "member").all()
flt.head()


### Cwiczenie 3.12.2 ⭐ - Mini-benchmark: petla vs wektoryzacja

Porownaj czas policzenia przychodu (`revenue`) dwoma sposobami:
- wolna funkcja `revenue_loop(...)` z pliku core,
- wersja wektorowa w NumPy/Pandas.

Zwracaj `Series` z:
- `loop_sec`
- `vectorized_sec`
- `speedup_x`

**Wymagane metody**
- `time_one`,
- wektoryzacja na kolumnach / tablicach.


In [ ]:
def benchmark_revenue_vectorization(trips: pd.DataFrame) -> pd.Series:
    # TODO:
    # Porównaj czas działania dwóch wersji liczenia revenue:
    # 1) wersji pętlowej
    # 2) wersji wektorowej
    #
    # Wynik ma być pd.Series z polami:
    # - "loop_sec"
    # - "vectorized_sec"
    # - "speedup_x"
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) najpierw policz czas przejazdu w minutach
    #    na podstawie:
    #    - "end_ts"
    #    - "start_ts"
    #
    #    podpowiedź:
    #    duration_min = (
    #        pd.to_datetime(trips["end_ts"]) - pd.to_datetime(trips["start_ts"])
    #    ).dt.total_seconds() / 60.0
    #
    # 2) pobierz kolumnę "user_type"
    #    podpowiedź:
    #    user_type = trips["user_type"]
    #
    # 3) zdefiniuj funkcję revenue_vectorized(...)
    #    która działa bez pętli, na całych tablicach naraz
    #
    #    W środku:
    #    - rozpoznaj, które wiersze to member
    #    - wyznacz base zależne od typu użytkownika
    #    - wyznacz per_min zależne od typu użytkownika
    #    - policz dopłatę tylko za minuty ponad 5
    #
    #    podpowiedzi:
    #    is_member = user_type.eq("member").to_numpy()
    #    base = np.where(is_member, 1.0, 2.5)
    #    per_min = np.where(is_member, 0.12, 0.23)
    #    np.maximum(duration_min.to_numpy() - 5.0, 0.0)
    #
    # 4) zmierz czas wersji pętlowej
    #    - użyj gotowej funkcji revenue_loop
    #    - użyj gotowej funkcji time_one(...)
    #    - do wersji pętlowej najlepiej przekazać tablice NumPy
    #
    #    podpowiedź:
    #    time_one(revenue_loop, duration_min.to_numpy(), user_type.to_numpy(), repeat=5)
    #
    # 5) zmierz czas wersji wektorowej
    #    - również użyj time_one(...)
    #
    #    podpowiedź:
    #    time_one(revenue_vectorized, duration_min, user_type, repeat=5)
    #
    # 6) policz przyspieszenie:
    #    speedup_x = loop_sec / vectorized_sec
    #
    # 7) zwróć pd.Series ze słownika:
    #    {
    #        "loop_sec": ...,
    #        "vectorized_sec": ...,
    #        "speedup_x": ...,
    #    }
    raise NotImplementedError


bench_rev = benchmark_revenue_vectorization(trips_small)
assert {"loop_sec", "vectorized_sec", "speedup_x"}.issubset(bench_rev.index)
assert bench_rev["loop_sec"] > 0
assert bench_rev["vectorized_sec"] > 0
assert bench_rev["speedup_x"] >= 1
bench_rev

### Cwiczenie 3.12.3 ⭐ - Mini-benchmark: maska boolean vs `query()`

Porownaj dwa sposoby filtrowania tych samych danych:
- klasyczna maska boolean,
- `DataFrame.query(...)`.

Uzyj tych samych warunkow co w Cwiczeniu 3.12.1.

Zwracaj `Series` z:
- `mask_sec`
- `query_sec`
- `same_n_rows`

**Wymagane metody**
- maska boolean,
- `query`,
- `time_one`.


In [ ]:
def benchmark_mask_vs_query(merged_feat: pd.DataFrame) -> pd.Series:
    # TODO:
    # Porównaj czas dwóch sposobów filtrowania danych:
    # 1) klasycznej maski boolean
    # 2) metody query()
    #
    # Oba sposoby mają wybierać dokładnie te same wiersze:
    # - 10 <= duration_min <= 45
    # - precip_mm <= 1.0
    # - user_type == "member"
    #
    # Wynik ma być pd.Series z polami:
    # - "mask_sec"
    # - "query_sec"
    # - "same_n_rows"
    #
    # PSEUDOKOD + wskazówki:
    #
    # 1) zapisz warunek dla query jako tekst
    #    podpowiedź:
    #    expr = "10 <= duration_min <= 45 and precip_mm <= 1.0 and user_type == 'member'"
    #
    # 2) zdefiniuj funkcję mask_filter(df)
    #    - zbuduj maskę logiczną przez:
    #      .between(...)
    #      .le(...)
    #      .eq(...)
    #    - potem wybierz wiersze przez .loc[mask]
    #
    #    podpowiedź:
    #    mask = (
    #        df["duration_min"].between(10, 45)
    #        & df["precip_mm"].le(1.0)
    #        & df["user_type"].eq("member")
    #    )
    #
    # 3) zdefiniuj funkcję query_filter(df)
    #    - użyj:
    #      df.query(expr)
    #
    # 4) zmierz czas obu funkcji przez time_one(...)
    #    podpowiedzi:
    #    mask_sec = time_one(mask_filter, merged_feat, repeat=5)
    #    query_sec = time_one(query_filter, merged_feat, repeat=5)
    #
    # 5) sprawdź, czy obie metody zwracają tę samą liczbę wierszy
    #    podpowiedź:
    #    len(mask_filter(merged_feat)) == len(query_filter(merged_feat))
    #
    # 6) zwróć wynik jako pd.Series:
    #    {
    #        "mask_sec": ...,
    #        "query_sec": ...,
    #        "same_n_rows": ...,
    #    }
    raise NotImplementedError


bench_filter = benchmark_mask_vs_query(merged_feat)
assert {"mask_sec", "query_sec", "same_n_rows"}.issubset(bench_filter.index)
assert bool(bench_filter["same_n_rows"]) is True
bench_filter

## ⭐ Add-on - przygotowanie do projektu


### Cwiczenie A.1 - Porownaj pamiec: `object` vs `category`

Dla kolumn `user_type` i `district` porownaj zuzycie pamieci przed i po konwersji na `category`.
Zapisz krotki wniosek tekstowy w zmiennej `MEMORY_NOTE`.


In [ ]:
merged_cat = merged_feat.copy()
before = merged_cat[["user_type", "district"]].memory_usage(deep=True).sum()

# TODO:
# 1) skonwertuj kolumnę "user_type" na typ category
# 2) skonwertuj kolumnę "district" na typ category
# 3) po konwersji policz ponownie zużycie pamięci tych dwóch kolumn
#
# podpowiedzi:
# merged_cat["user_type"] = merged_cat["user_type"].astype("category")
# merged_cat["district"] = merged_cat["district"].astype("category")
# after = merged_cat[["user_type", "district"]].memory_usage(deep=True).sum()

after = None

MEMORY_NOTE = (
    f"Po konwersji kolumn tekstowych na `category` zużycie pamięci spadło z {before} do {after} bajtów. "
    "Dzieje się tak dlatego, że Pandas przechowuje słownik unikalnych etykiet i lekkie kody liczbowe "
    "zamiast powielać pełne napisy w każdym wierszu. "
    "To szczególnie opłaca się dla kolumn z niewielką liczbą powtarzających się wartości, takich jak "
    "`user_type` i `district`."
)

assert after is not None and after <= before
assert isinstance(MEMORY_NOTE, str) and len(MEMORY_NOTE.strip()) >= 80
before, after, MEMORY_NOTE